# MLTrail — demo

Step-by-step walkthrough of the MLTrail model registry: register models, version them,
list / search / trail, predict on new compounds, then overwrite and delete.
**Public SMILES + synthetic models only** — no project data. Run top to bottom.


## Setup
Consolidated imports + autoreload so edits to `mltrail/*.py` are picked up live.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # import mltrail from the repo root

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from mltrail import Registry
from mltrail.featurizers import get_featurizer

## Config
An isolated temp registry; featurizers are reused from `Rdkit_tools` (train/predict parity).

In [ ]:
demo_dir = Path(tempfile.mkdtemp())
config = {
    "registry_path": str(demo_dir / "registry.json"),
    "date_format": "%Y%m%d_%H%M%S",
    "featurizers": {"path": "/home/gtamo/Scripts", "module": "Rdkit_tools",
                    "map": {"MF_2048": "get_MF_bits_from_df", "H236": "compute_H236_features"}},
}
reg = Registry.from_config(config)
demo_dir

## Train two tiny models
Public molecules, random labels, saved in the `{'model','feature_cols',...}` dict format MLTrail expects.

In [ ]:
train = pd.DataFrame({"compound": [f"C_{i}" for i in range(5)],
                      "smiles": ["CCO", "c1ccccc1", "CC(=O)Oc1ccccc1C(=O)O", "CCN(CC)CC", "CCOCC"]})
X = get_featurizer("MF_2048", config)(train).drop(columns="compound")
np.random.seed(0)

reg_model = RandomForestRegressor(n_estimators=8, random_state=0).fit(X, np.random.rand(len(X)))
reg_path = demo_dir / "solubility_rf.joblib"
joblib.dump({"model": reg_model, "feature_cols": list(X.columns), "features": "MF_2048"}, reg_path)

clf_model = RandomForestClassifier(n_estimators=8, random_state=0).fit(X, np.random.randint(0, 2, len(X)))
clf_path = demo_dir / "tox_rf.joblib"
joblib.dump({"model": clf_model, "feature_cols": list(X.columns), "features": "MF_2048"}, clf_path)
print("saved:", reg_path.name, "and", clf_path.name)

## Register models  (`--add`)
First-time registration assigns an auto-incrementing id.

In [ ]:
id_sol = reg.add(experiment_name="LogS_MF", experiment_measure="solubility", unit="logS",
                 model_path=str(reg_path), model_type="single_task_regression",
                 framework="sklearn", features_type="MF_2048", metrics={"R2": 0.81})
id_tox = reg.add(experiment_name="hERG_tox", experiment_measure="toxicity", unit="binary",
                 model_path=str(clf_path), model_type="single_task_classification",
                 framework="sklearn", features_type="MF_2048", metrics={"roc_auc": 0.88})
id_sol, id_tox

## Add a new version  (`--add --id`)
A retrain appends a version under the same id; identity fields are inherited.

In [ ]:
reg.add(model_id=id_sol, model_path=str(reg_path), metrics={"R2": 0.85})
print("latest version of model", id_sol, "->", reg.details(id_sol)["version"])

## List  (`--list`)
One row per model (latest version), sorted alphabetically by experiment_name.

In [ ]:
reg.list()

## Details  (`--details --id`)
Every attribute of a model's latest version.

In [ ]:
reg.details(id_sol)

## Search  (`--search`)
Match ANY provided field (case-insensitive substring).

In [ ]:
reg.search(experiment_measure="tox")

## Trail  (`--trail --metrics R2`)
A metric across versions, ready to plot metric-vs-date.

In [ ]:
reg.trail("R2", model_id=id_sol)

## Predict  (`--predict`)
Score a dataset of public molecules; output is smiles, compound, prediction.

In [ ]:
to_predict = demo_dir / "new_cmps.csv"
pd.DataFrame({"cid": ["P1", "P2", "P3"], "smi": ["CCO", "c1ccccc1", "CC(=O)O"]}).to_csv(to_predict, index=False)
preds = reg.predict(id_sol, str(to_predict), smiles_column="smi", compound_id="cid")
preds

## Overwrite + delete  (`--overwrite`, `--delete`)
`--overwrite` resets the latest version in place; `--delete` removes a model and its whole trail.

In [ ]:
reg.add(model_id=id_tox, overwrite=True, model_path=str(clf_path), metrics={"roc_auc": 0.91})
reg.delete(id_tox)
reg.list()

## CLI equivalents

The same flow from a shell (after `pip install -e . --no-deps` into the `ML` env):

```bash
mltrail --add --experiment_name LogS_MF --experiment_measure solubility --unit logS \
        --model_path solubility_rf.joblib --model_type single_task_regression \
        --framework sklearn --features_type MF_2048 --metrics R2=0.81
mltrail --add --id 1 --model_path solubility_rf.joblib --metrics R2=0.85   # new version
mltrail --list
mltrail --details --id 1
mltrail --search --experiment_measure solubility
mltrail --trail --metrics R2 --id 1 --output_trail trail.csv
mltrail --predict --id 1 --dataset new_cmps.csv --smiles_column smi --compound_id cid --pred_output preds.csv
mltrail --overwrite --id 1 --model_path solubility_rf.joblib --metrics R2=0.9
mltrail --delete --id 2
```

## Assertions
Separate from the run cells above, so they can be re-checked without re-running the demo.

In [ ]:
listing = reg.list()
# only the solubility model remains after deleting the toxicity model
assert list(listing["experiment_name"]) == ["LogS_MF"]
# the solubility model accumulated two versions
assert reg.details(id_sol)["version"] == 2
# prediction produced the spec columns, one row per input compound
assert list(preds.columns) == ["smiles", "compound", "prediction"] and len(preds) == 3
print("demo assertions passed")